In [1]:
import yaml
import numpy as np 
from pytorch_lightning import Trainer
import importlib

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = 'config/commonvoice/cv_word_baseline.yaml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [3]:
config

{'corpus': {'root': '/om2/user/imgriff/datasets/commonvoice_9_en/3000ms/stimSR_50000/cv_9_en/subsets/'},
 'audio': {'rep_type': 'cochlea_filt',
  'rep_kwargs': {'sr': 50000,
   'env_sr': 10000,
   'n_channels': 40,
   'low_lim': 40,
   'use_pad': True,
   'rep_on_gpu': True,
   'center_crop': True,
   'out_dur': 2,
   'impulse_len': 1,
   'env_extraction_type': 'Half-wave Rectification',
   'downsampling_type': 'TorchTransformsResample',
   'downsampling_kwargs': {'lowpass_filter_width': 64,
    'rolloff': 0.9475937167399596,
    'resampling_method': 'kaiser_window',
    'beta': 14.769656459379492}},
  'compression_type': 'coch_p3',
  'compression_kwargs': {'scale': 1,
   'offset': 1e-07,
   'clip_value': 5,
   'power': 0.3}},
 'loader': {'batch_size': 512},
 'model': {'num_words': 998, 'fc_size': 512},
 'val_metric': 'ACC/val_acc',
 'model_name': 'cv_clean_word_rec_baseline',
 'hparas': {'valid_step': 500,
  'epochs': 10,
  'optimizer': 'Adam',
  'lr': 0.0001,
  'eps': 1e-08,
  'lr_sc

In [4]:
!nvidia-smi

Thu May 25 20:38:43 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 515.86.01    Driver Version: 515.86.01    CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Quadro RTX 6000     On   | 00000000:3B:00.0 Off |                  Off |
| 37%   52C    P8    11W / 260W |      1MiB / 24576MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [5]:
config['loader']['num_workers'] = 0
config['loader']['batch_size'] = 32
config['corpora_name'] = 'TIMIT'
config['audio']['rep_kwargs']['rep_on_gpu'] = True
config['corpus']['root']  = '/om2/user/imgriff/datasets/timit/clean_timit_targets_attn_task_0.1rms.pdpkl'

ckpt_path =  "attn_cue_models/cv_baseline_word_task/checkpoints/epoch=8-step=18416-v2.ckpt"


In [6]:
# sys.path.append('../')
from src import cv_word_lightning
importlib.reload(cv_word_lightning)

CommonVoiceWordRec = cv_word_lightning.CommonVoiceWordRec

model = CommonVoiceWordRec.load_from_checkpoint(checkpoint_path=ckpt_path, config=config)


Eval on TIMIT stimuli
center_crop=True
binaural=False
using FIR cochleagram


In [7]:
import pickle 
class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_word_int_label_dict.pkl", "rb" )) 


In [8]:
from corpus import timit 
importlib.reload(timit)
eval_stim_path = '/om2/user/imgriff/datasets/timit/clean_timit_targets_attn_task_0.1rms.pdpkl'
dataset = timit.TIMIT_CV_Compat_Prepaired(eval_stim_path, clean_targets=True)

In [9]:
from IPython.display import Audio

In [10]:
wav, word_ix = dataset[1]

print(dataset.cv_class_map[word_ix])
Audio(wav, rate=50_000)

novel


In [11]:
trainer = Trainer(
    precision=32,
    default_root_dir='test_log_dump/',
    deterministic=True,
    val_check_interval=1,
#     limit_train_batches=0.,
    # limit_val_batches=0.0,
    num_nodes=1,
    gpus=1,
    accelerator="gpu"
)

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/loops/utilities.py:91: PossibleUserWarning: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
  rank_zero_warn(
GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
`Trainer(val_check_interval=1)` was configured so validation will run after every batch.


In [12]:

trainer.test(model)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:240: PossibleUserWarning: The dataloader, test_dataloader 0, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 80 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Test set length =  902
Testing DataLoader 0:   3%|▎         | 1/29 [00:05<02:36,  5.59s/it]

2023-05-25 20:39:10.309963: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-05-25 20:39:11.793310: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Testing DataLoader 0: 100%|██████████| 29/29 [01:44<00:00,  3.59s/it]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   ACC/test_acc_epoch       0.6040171980857849
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'ACC/test_acc_epoch': 0.6040171980857849}]

In [ ]:
import pandas as pd 
model_results_path = "attn_cue_models/cv_word_rec_pilot/TIMIT_task_clean_cv_clean_word_rec_baseline/version_3/metrics.csv"
model_results = pd.read_csv(model_results_path)

In [50]:
model_results

,Losses/test_loss,ACC/test_acc_step,step,ACC/test_acc_epoch,epoch
0,11.703489,0.375000,0,NaN,NaN
1,7.365752,0.312500,1,NaN,NaN
2,7.545806,0.437500,2,NaN,NaN
3,9.537827,0.437500,3,NaN,NaN
4,11.516878,0.406250,4,NaN,NaN
5,6.772373,0.562500,5,NaN,NaN
6,10.780080,0.375000,6,NaN,NaN
7,7.852668,0.437500,7,NaN,NaN
8,6.582730,0.531250,8,NaN,NaN
9,9.580922,0.500000,9,NaN,NaN
